# Managed RAG with Bedrock Knowledge Bases (S3 Vectors)

In `06_question_answering` we grounded the model by attaching a document to each request. That works for a few small files, but it doesn't scale - you can't stuff a whole document library into every prompt.

A **Knowledge Base** is managed RAG. You point Bedrock at documents in S3 and it chunks them, generates embeddings, and stores them in a vector index. At query time it retrieves only the relevant chunks and uses them to ground the answer.

> **Teaching/Learning Tip:** the slides use **OpenSearch Serverless** as the vector store. We use **S3 Vectors** instead - it scales to zero and bills per storage/query, so a demo costs cents. An idle OpenSearch Serverless collection runs ~$350/month. Same Knowledge Base API either way; only the `storageConfiguration` differs.

## Provisioning is done by `setup_kb.py`

Standing up a Knowledge Base means creating several resources in order (vector store, S3 data bucket, IAM role, the KB, a data source, and an ingestion job). That's a lot of slow, stateful setup that doesn't belong in a notebook, so it lives in `setup_kb.py`:

```bash
python setup_kb.py            # create everything + ingest the sample doc
python setup_kb.py --cleanup  # tear it all down (do this when finished!)
```

This notebook assumes you've run `setup_kb.py` and picks up the knowledge base id it saved.

The three slides map to these steps in `setup_kb.py`:
- **CreateKnowledgeBase** -> `create_knowledge_base()` (with `S3_VECTORS` storage)
- **CreateDataSource** -> `create_data_source()` (S3 connector, fixed-size chunking)
- **StartIngestionJob** -> `start_ingestion()`

## The chunking configuration (slide 2)

The data source controls how documents are split before embedding. Fixed-size chunking with a small overlap is a common default:

```python
vectorIngestionConfiguration={
    "chunkingConfiguration": {
        "chunkingStrategy": "FIXED_SIZE",
        "fixedSizeChunkingConfiguration": {
            "maxTokens": 100,
            "overlapPercentage": 10,
        },
    }
}
```

> **Teaching/Learning Tip:** chunk size is a tradeoff. Small chunks give precise retrieval but can lose context; large chunks keep context but dilute relevance. The overlap keeps a sentence from being split awkwardly across two chunks.

> **Gotcha (learned the hard way):** the slide's CLI sets `inclusionPrefixes` to `".*\\.pdf"`. That looks like a regex, but the S3 connector treats `inclusionPrefixes` as **literal key prefixes**, not regex - so it silently matches *nothing* and ingestion indexes 0 documents (the job still reports COMPLETE!). If you want to filter, use a real prefix like `"docs/"`. `setup_kb.py` omits it so the whole bucket is ingested.

## Query it: the RetrieveAndGenerate API

This is the payoff. One call embeds the question, searches the vector index, feeds the matching chunks to a model, and returns a grounded answer **with citations**.

We wrap it in a small `retrieveAndGenerate(input, kbId)` helper - the same shape as the slide - so the call is a one-liner.

> **Note on the model ARN:** the slide shows `...:foundation-model/anthropic.haiku-v1`, but Claude models can't be called on-demand by a bare foundation-model ARN - they need an *inference profile*. We use a Nova Lite inference profile here, which works for generation. The KB and API call are otherwise identical to the slide.

In [ ]:
import json
import boto3

REGION = "us-east-1"
kb_id = json.load(open("kb_config.json"))["knowledgeBaseId"]
account = boto3.client("sts", region_name=REGION).get_caller_identity()["Account"]
bedrock_agent_runtime = boto3.client("bedrock-agent-runtime", region_name=REGION)

MODEL_ARN = f"arn:aws:bedrock:{REGION}:{account}:inference-profile/us.amazon.nova-lite-v1:0"


def retrieveAndGenerate(input, kbId):
    return bedrock_agent_runtime.retrieve_and_generate(
        input={"text": input},
        retrieveAndGenerateConfiguration={
            "type": "KNOWLEDGE_BASE",
            "knowledgeBaseConfiguration": {
                "knowledgeBaseId": kbId,
                "modelArn": MODEL_ARN,
            },
        },
    )


question = (
    "Which vendor has the largest market share, and what is the primary "
    "bottleneck for the industry?"
)

response = retrieveAndGenerate(question, kb_id)
print(response["output"]["text"])

The citations tell you which source document(s) the answer came from - essential for trust and for letting users verify claims.

In [ ]:
for citation in response.get("citations", []):
    for ref in citation.get("retrievedReferences", []):
        uri = ref.get("location", {}).get("s3Location", {}).get("uri", "?")
        snippet = ref["content"]["text"][:120]
        print(f"Source: {uri}")
        print(f"  chunk: {snippet}...\n")

## Cleanup

When you're done, tear everything down so nothing keeps billing:

```bash
python setup_kb.py --cleanup
```

> **Teaching/Learning Tip:** S3 Vectors is cheap, but always clean up demo infrastructure. The cleanup removes the KB, data source, IAM role, both S3 buckets, and the vector index/bucket - in the right order (dependencies first).